In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import mlflow
import pandas as pd
import re


# Pipeline de sentiment
classifier = pipeline("sentiment-analysis",
                      model="distilbert-base-uncased-finetuned-sst-2-english",
                      truncation=True   # tronque automatiquement les textes trop longs
)


Device set to use cuda:0


In [ ]:
def nettoyer_texte(text):
    text = re.sub(r"<.*?>", "", text)  # remove HTML tags
    text = re.sub(r"\s+", " ", text)   # remove multiple spaces
    text = text.strip()
    return text
df = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")
df['review'] = df['review'].apply(nettoyer_texte)
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:


mlflow.set_experiment("IMDB_Sentiment_Analysis_Bert")

# Nettoyage et prédiction
with mlflow.start_run(run_name="DistilBERT_Inference") as run:
    results = classifier(list(df["review"]), batch_size=16, truncation=True)
    df['sentiment_label'] = [r['label'].strip().upper() if r['label'] is not None else None for r in results]
    df['sentiment_score'] = [r['score'] for r in results]

    # Nettoyage labels vrais
    df['sentiment'] = df['sentiment'].astype(str).str.strip().str.upper()

    # Supprimer toutes les lignes où y_true ou y_pred est NaN ou invalide
    valid_labels = {"POSITIVE", "NEGATIVE"}
    df_clean = df[df['sentiment'].isin(valid_labels) & df['sentiment_label'].isin(valid_labels)].copy()

    # Map binaire
    label_map = {"NEGATIVE": 0, "POSITIVE": 1}
    y_true = df_clean['sentiment'].map(label_map).values
    y_pred_bin = df_clean['sentiment_label'].map(label_map).values

    # Metrics
    accuracy = accuracy_score(y_true, y_pred_bin)
    precision = precision_score(y_true, y_pred_bin)
    recall = recall_score(y_true, y_pred_bin)
    f1 = f1_score(y_true, y_pred_bin)

    mlflow.log_metric("avg_confidence", df_clean['sentiment_score'].mean())
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.log_text("\n".join([f"{lab}: {scr}" for lab, scr in zip(df_clean['sentiment_label'][:100], df_clean['sentiment_score'][:100])]), "sample_predictions.txt")

    print(f"Run MLflow terminé, run_id: {run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")


Run MLflow terminé, run_id: 474af402de584f4c9faef347368127a1
Accuracy: 0.9002, Precision: 0.9341, Recall: 0.8646, F1: 0.8980


In [1]:
from transformers import pipeline

c:\Users\Abdessamad\Documents\NLP\Abdessamad_AMTOUG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    "sentiment-analysis",
    model="../model/distilbert_model",
    device=-1  
)

)

classifier.save_pretrained("../model/distilbert_model")
